# Feature Aggregation for Agent-Based Crop Health Simulation

Convert the processed features into nine model inputs (`i1` to `i9`) that represent the main support and stress conditions affecting crop health.

## Workflow Focus

`prepared dataset -> composite support/stress indicators -> simulation-ready feature table`


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd

from src.data_preprocessing import preprocess_csv
from src.feature_aggregation import I_TERM_COLUMNS, add_aggregated_features


In [2]:
prepared_path = PROJECT_ROOT / "data" / "prepared_data_for_dynamic_model.csv"
if prepared_path.exists():
    prepared_df = pd.read_csv(prepared_path)
else:
    prepared_df = preprocess_csv(PROJECT_ROOT / "data" / "Smart Farming Dataset 2024.csv").prepared_df

print(prepared_df.shape)
prepared_df.head()


(2200, 46)


,label,soil_type,water_source_type,growth_stage,n_scaled,p_scaled,k_scaled,temperature_scaled,humidity_scaled,ph_scaled,...,opt_humidity,opt_moisture,opt_rainfall,opt_ph,opt_sunlight,stress_tol,growth_rate,water_need,vpd_tolerance,fertility_need
0,rice,2,3,1,0.642857,0.264286,0.190,0.345886,0.790267,0.466264,...,0.793417,0.458182,0.775896,0.454209,0.552827,0.510234,0.406507,0.458182,0.872388,0.38085
1,rice,3,2,1,0.607143,0.378571,0.180,0.371445,0.770633,0.549480,...,0.793417,0.458182,0.775896,0.454209,0.552827,0.510234,0.406507,0.458182,0.872388,0.38085
2,rice,2,2,1,0.428571,0.357143,0.195,0.406854,0.793977,0.674219,...,0.793417,0.458182,0.775896,0.454209,0.552827,0.510234,0.406507,0.458182,0.872388,0.38085
3,rice,3,3,3,0.528571,0.214286,0.175,0.506901,0.768751,0.540508,...,0.793417,0.458182,0.775896,0.454209,0.552827,0.510234,0.406507,0.458182,0.872388,0.38085
4,rice,2,3,2,0.557143,0.264286,0.185,0.324378,0.785626,0.641291,...,0.793417,0.458182,0.775896,0.454209,0.552827,0.510234,0.406507,0.458182,0.872388,0.38085


## Aggregate Interpretable Indicators

The aggregated terms are kept explicit so the report clearly shows how nutrient balance, soil context, atmospheric support, water deficit, growth readiness, and management context enter the model.


In [3]:
feature_df = add_aggregated_features(prepared_df)
print(feature_df.shape)
feature_df[["label", *I_TERM_COLUMNS]].head()


(2200, 55)


,label,i1_nutrient_level_balance,i2_soil_context,i3_atmospheric_support,i4_atmospheric_stress,i5_water_support,i6_water_deficit,i7_growth_readiness,i8_biotic_env_stress,i9_management_context
0,rice,0.539383,0.587200,0.945403,0.236283,0.721875,0.189066,0.380570,0.728357,0.487682
1,rice,0.571272,0.591433,0.771467,0.251869,0.703310,0.595929,0.339242,0.757581,0.116275
2,rice,0.560249,0.587033,0.811743,0.264658,0.718020,0.176276,0.393538,0.078601,0.777569
3,rice,0.509410,0.701363,0.804145,0.319413,0.729335,0.251573,0.614362,0.475780,0.127022
4,rice,0.532489,0.648711,0.876004,0.226955,0.720626,0.221616,0.435000,0.334512,0.922823


## Quick Inspection

A compact descriptive summary helps verify the range and spread of the nine indicators before simulation.


In [4]:
feature_df[I_TERM_COLUMNS].describe().T


,count,mean,std,min,25%,50%,75%,max
i1_nutrient_level_balance,2200.0,0.504737,0.100242,0.263425,0.440842,0.488377,0.550462,0.784881
i2_soil_context,2200.0,0.604113,0.067572,0.358786,0.557224,0.605601,0.653327,0.772088
i3_atmospheric_support,2200.0,0.822743,0.065040,0.611698,0.777387,0.823812,0.871063,0.986094
i4_atmospheric_stress,2200.0,0.340385,0.097979,0.079102,0.279608,0.322756,0.388089,0.726211
i5_water_support,2200.0,0.649768,0.071020,0.397878,0.605341,0.659630,0.701719,0.802958
i6_water_deficit,2200.0,0.435843,0.158208,0.117080,0.297614,0.436089,0.568947,0.759485
i7_growth_readiness,2200.0,0.521403,0.108575,0.224504,0.448416,0.521078,0.596939,0.895348
i8_biotic_env_stress,2200.0,0.500233,0.207871,0.006062,0.344470,0.493353,0.651706,0.990013
i9_management_context,2200.0,0.500465,0.207369,0.008293,0.352238,0.499426,0.649509,0.985849


## Save Simulation Inputs

The resulting table becomes the direct input for the agent-based crop health model.


In [5]:
feature_output = PROJECT_ROOT / "data" / "features_for_simulation.csv"
feature_df.to_csv(feature_output, index=False)
print(f"Saved simulation-ready feature table to: {feature_output}")


Saved simulation-ready feature table to: C:\Users\banya\OneDrive\Documents\GitHub\Crop-Health-ABM\data\features_for_simulation.csv
